# Optional - Continuous Kafka to Bronze Delta

Start the Docker stack and producer first. This PySpark Structured Streaming query appends Kafka events to the same bronze Delta table used by batch ingestion.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None

from insightsstream.delta_pipeline import DeltaSettings, create_delta_spark_session, start_kafka_to_bronze

settings = DeltaSettings.from_env()
spark = create_delta_spark_session("InsightsStream-Kafka-Delta", settings)

In [ ]:
query = start_kafka_to_bronze(spark, settings, available_now=True)
query.awaitTermination()
print(f"Kafka events appended to {settings.uri('bronze/events')}")

In [ ]:
bronze = spark.read.format("delta").load(settings.uri("bronze/events"))
bronze.groupBy("source_type").count().show()
bronze.orderBy("ingested_at", ascending=False).show(20, truncate=False)